# Task 2: Grouped Train/Test Split (80/20 by province)

Purpose: seal away ~15 entire provinces as a one-time final test set.
- Group column: province (all rows of a province stay together)
- The test set is real data only, forever; it is never augmented
- The test set is evaluated exactly once, at the end, by Member 3
- All model comparison happens inside the training provinces via LOPO CV
Outputs: artifacts/split_provinces.json and artifacts/train_test_split_seed.txt

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

DATA_PATH = Path("../data/processed/merged_rice_yield_dataset.csv")
ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
groups = df["province"]

SPLIT_SEED = 42
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SPLIT_SEED)
train_idx, test_idx = next(gss.split(df, groups=groups))

train_provinces = sorted(groups.iloc[train_idx].unique())
test_provinces = sorted(groups.iloc[test_idx].unique())

print(f"Train: {len(train_idx)} rows across {len(train_provinces)} provinces")
print(f"Test:  {len(test_idx)} rows across {len(test_provinces)} provinces")
print(f"Province overlap between train and test: {len(set(train_provinces) & set(test_provinces))}")
print("\nHeld-out test provinces:")
print(test_provinces)

Train: 229 rows across 60 provinces
Test:  58 rows across 16 provinces
Province overlap between train and test: 0

Held-out test provinces:
['Abra', 'Albay', 'Benguet', 'Bohol', 'Camiguin', 'Dinagat Islands', 'Ilocos Sur', 'Isabela', 'Misamis Oriental', 'Negros Occidental', 'Northern Samar', 'Occidental Mindoro', 'Quezon', 'South Cotabato', 'Sultan Kudarat', 'Tarlac']


In [2]:
import json

split_info = {
    "method": "GroupShuffleSplit(n_splits=1, test_size=0.2)",
    "group_column": "province",
    "random_seed": SPLIT_SEED,
    "n_train_rows": int(len(train_idx)),
    "n_test_rows": int(len(test_idx)),
    "train_provinces": train_provinces,
    "test_provinces": test_provinces,
}

with open(ARTIFACTS_DIR / "split_provinces.json", "w") as f:
    json.dump(split_info, f, indent=2)

seed_note = f"""Train/test split documentation
==============================
Method: GroupShuffleSplit(n_splits=1, test_size=0.2), scikit-learn
Group column: province (all rows of a province stay together)
Random seed: {SPLIT_SEED}
Train: {len(train_idx)} rows / {len(train_provinces)} provinces
Test:  {len(test_idx)} rows / {len(test_provinces)} provinces
Rule: test set is real data only, never augmented, evaluated once at the end.

Test provinces:
{', '.join(test_provinces)}

Train provinces:
{', '.join(train_provinces)}
"""

with open(ARTIFACTS_DIR / "train_test_split_seed.txt", "w") as f:
    f.write(seed_note)

print("Saved:", (ARTIFACTS_DIR / "split_provinces.json").resolve())
print("Saved:", (ARTIFACTS_DIR / "train_test_split_seed.txt").resolve())

Saved: C:\Users\Charles\Documents\rice-yield-prediction\artifacts\split_provinces.json
Saved: C:\Users\Charles\Documents\rice-yield-prediction\artifacts\train_test_split_seed.txt


## Task 2 Summary: Grouped Train/Test Split
- **Method:** `GroupShuffleSplit(test_size=0.2, random_state=42)` grouped by `province`.
- **Training Set:** 229 rows across 60 provinces. Used for LOPO CV, augmentation, and model training.
- **Test Set:** 58 rows across 16 provinces. Held out entirely. Never augmented, evaluated exactly once at the end by Member 3.
- **Leakage Check:** Confirmed 0 province overlap between train and test sets.
- **Artifacts Saved:** 
  - `artifacts/split_provinces.json` (Exact lists of train/test provinces)
  - `artifacts/train_test_split_seed.txt` (Human-readable split documentation)